<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 15. Transformer 구조 해부 실습 — 직접 만드는 causal Transformer

이 파일은 **실습용/정답용이 같은 셀 순서를 갖는 페어 노트북**입니다. JupyterLab에서
왼쪽에는 `exercises/15_...`, 오른쪽에는 `solutions/15_...`를 열고 셀 단위로 따라 치세요.

이번 랩의 목표는 완제품 API를 호출하는 것이 아니라 `[B, T, D]` 텐서가 Q/K/V,
head 분리, mask, residual을 통과하는 과정을 눈으로 확인하는 것입니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 완료 기준과 권장 순서

1. Token/position embedding을 더한다.
2. Q/K/V projection과 scaled dot-product attention을 구현한다.
3. `[B,T,D] ↔ [B,H,T,Dh]` 변환과 causal/padding mask를 검증한다.
4. Pre-norm residual, FFN, decoder block, 작은 causal LM을 조립한다.
5. shape, gradient, 미래 토큰 누설 방지 assert를 모두 통과시킨다.

**학습법:** 정답 전체를 복사하지 말고 TODO 하나를 먼저 작성한 뒤 해당 검증 셀만 실행하세요.

In [ ]:
# 준비 셀: 이 셀은 그대로 실행해도 됩니다.
import math
import random
from dataclasses import dataclass

import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 15
random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(f"{ACCELERATOR.summary()}, torch={torch.__version__}")

## 1) 미니 배치와 vocabulary

`input_ids`의 축은 `[batch, time]`입니다. `<pad>`는 문장 길이를 맞추기 위한 토큰이고,
`valid_tokens`가 False인 위치는 key로도 query로도 attention에 참여시키지 않습니다.
실제 서비스에서는 tokenizer가 이 과정을 담당하지만 구조 학습을 위해 작은 단어 사전을 직접 만듭니다.

In [ ]:
# TODO 15-1: 아래 토큰으로 stoi/itos를 만들고 두 문장을 정수 텐서로 바꾸세요.
token_rows = [
    ["<bos>", "나는", "모델을", "학습한다", "<eos>", "<pad>"],
    ["<bos>", "attention은", "문맥을", "연결한다", "<eos>", "<pad>"],
]
vocab_tokens = ["<pad>", "<bos>", "<eos>"] + sorted(
    {token for row in token_rows for token in row if not token.startswith("<")}
)
# TODO: stoi, itos, input_ids, pad_id, valid_tokens를 완성하세요.
stoi = ...
itos = ...
input_ids = ...
pad_id = ...
valid_tokens = ...

## 2) Token embedding + learned position embedding

같은 단어라도 순서가 다르면 의미가 달라집니다. Token embedding은 **무엇인지**, position
embedding은 **어디에 있는지**를 표현하고 둘은 같은 `model_dim`이므로 원소별로 더할 수 있습니다.
출력 shape은 항상 `[B, T, D]`입니다.

In [ ]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size: int, model_dim: int, max_length: int, pad_id: int):
        super().__init__()
        # TODO 15-2: token/position nn.Embedding 두 개를 정의하세요.
        self.token = ...
        self.position = ...
        self.max_length = max_length

    def forward(self, ids: Tensor) -> Tensor:
        # TODO: positions=[0..T-1]을 batch에 맞춰 더하세요.
        raise NotImplementedError("TODO 15-2")

In [ ]:
# 검증: 구현 후 shape와 위치 변화가 표현에 반영되는지 확인하세요.
model_dim = 32
embedding = TokenPositionEmbedding(len(stoi), model_dim, 16, pad_id).to(device)
hidden = embedding(input_ids)
assert hidden.shape == (2, 6, model_dim)
assert not torch.allclose(hidden[:, 0], hidden[:, 1])
print("embedding:", tuple(hidden.shape), "dtype:", hidden.dtype)

## 3) Q, K, V projection

한 hidden vector를 세 관점으로 선형 변환합니다. Query는 "무엇을 찾는가", Key는
"나는 무엇을 나타내는가", Value는 "선택되면 무엇을 전달하는가"로 이해할 수 있습니다.
현업 구현은 메모리 접근을 줄이기 위해 보통 하나의 `Linear(D, 3D)` 결과를 세 덩어리로 나눕니다.

In [ ]:
# TODO 15-3: 하나의 projection 결과를 마지막 축에서 q, k, v로 나누세요.
qkv_projection = nn.Linear(model_dim, 3 * model_dim).to(device)
qkv = qkv_projection(hidden)
query, key, value = ...

## 4) Multi-head reshape

`D = H × Dh`로 나누어 여러 관계를 병렬로 봅니다. 연산을 위해 `[B,T,D]`를
`[B,H,T,Dh]`로 바꾸며, `transpose` 뒤에는 메모리가 비연속일 수 있으므로 합칠 때
`contiguous()`를 사용하는 습관이 중요합니다.

In [ ]:
def split_heads(x: Tensor, num_heads: int) -> Tensor:
    # TODO 15-4: [B,T,D] -> [B,H,T,Dh]
    raise NotImplementedError("TODO 15-4")


def merge_heads(x: Tensor) -> Tensor:
    # TODO 15-4: [B,H,T,Dh] -> [B,T,D]
    raise NotImplementedError("TODO 15-4")

In [ ]:
num_heads = 4
qh, kh, vh = (split_heads(t, num_heads) for t in (query, key, value))
assert qh.shape == (2, 4, 6, 8)
assert torch.allclose(merge_heads(qh), query)
print("per head:", tuple(qh.shape), "merged:", tuple(merge_heads(qh).shape))

## 5) Scaled dot-product attention

`scores = QKᵀ / sqrt(Dh)`입니다. 차원이 커지면 dot product 분산이 커져 softmax가
지나치게 뾰족해지므로 `sqrt(Dh)`로 나눕니다. Mask는 **softmax 전에** 허용되지 않은
score를 매우 작은 값으로 바꿉니다. 전부 mask된 padding query는 명시적으로 0으로 처리합니다.

In [ ]:
def scaled_dot_product_attention(
    q: Tensor, k: Tensor, v: Tensor, allowed: Tensor | None = None
) -> tuple[Tensor, Tensor]:
    # TODO 15-5: score 계산 -> mask -> softmax -> value 가중합 순서로 구현하세요.
    raise NotImplementedError("TODO 15-5")

## 6) Causal mask + padding mask

- Causal mask: query 위치 `t`가 `t+1` 이후 key를 보지 못하게 합니다.
- Padding key mask: `<pad>`를 정보원으로 선택하지 못하게 합니다.
- Padding query mask: 의미 없는 query 행의 attention을 0으로 만듭니다.

최종 `allowed`는 broadcast 가능한 `[B,1,T,T]` boolean 텐서입니다.

In [ ]:
def build_allowed_mask(valid: Tensor) -> Tensor:
    # TODO 15-6: tril causal mask와 key/query validity를 AND 하세요.
    raise NotImplementedError("TODO 15-6")

In [ ]:
allowed = build_allowed_mask(valid_tokens)
context, attention_weights = scaled_dot_product_attention(qh, kh, vh, allowed)
future = torch.ones(6, 6, dtype=torch.bool, device=device).triu(diagonal=1)
assert allowed.shape == (2, 1, 6, 6)
assert torch.all(attention_weights.masked_select(future.view(1, 1, 6, 6)) == 0)
assert torch.all(attention_weights[..., -1, :] == 0)  # padding query
assert torch.all(attention_weights[..., :, -1] == 0)  # padding key
print("context:", tuple(context.shape), "mask passed")

## 7) Attention을 사람이 읽을 수 있는 데이터로 보기

아래 `attention_view`는 시각화 라이브러리에 바로 전달할 수 있는 토큰 label과 2차원 weight
행렬입니다. 행은 query, 열은 key입니다. causal mask 때문에 대각선 오른쪽 값이 모두 0인지
확인하세요. 정답 노트북은 추가 설치 없이도 보이도록 텍스트 heatmap 형태로 출력합니다.

In [ ]:
# TODO 15-7: sample=0, head=0의 [T,T] weight와 token label을 묶으세요.
tokens_for_view = token_rows[0]
matrix_for_view = ...
attention_view = {"tokens": tokens_for_view, "weights": ...}

## 8) Multi-head self-attention 모듈

이제 흩어진 연산을 하나의 `nn.Module`로 묶습니다. `qkv` projection → head 분리 → attention
→ head 병합 → output projection 순서입니다. 학습/디버깅을 위해 weight도 함께 반환합니다.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        # TODO 15-8: divisibility 검증, qkv/out projection을 정의하세요.
        self.num_heads = num_heads
        self.qkv = ...
        self.out = ...

    def forward(self, x: Tensor, allowed: Tensor) -> tuple[Tensor, Tensor]:
        # TODO: qkv -> split -> attention -> merge -> out
        raise NotImplementedError("TODO 15-8")

In [ ]:
mha = MultiHeadSelfAttention(model_dim, num_heads).to(device)
mha_output, mha_weights = mha(hidden, allowed)
assert mha_output.shape == hidden.shape
assert mha_weights.shape == (2, num_heads, 6, 6)
print("MHA output:", tuple(mha_output.shape))

## 9) Pre-norm residual + Feed Forward Network

Pre-norm block은 `x + Attention(LN(x))`, 이어서 `x + FFN(LN(x))`를 수행합니다.
Residual 경로는 gradient가 깊은 네트워크를 통과하도록 돕고, FFN은 각 토큰 위치에 동일한
두 층 MLP를 적용합니다. 중간 차원은 보통 model dimension보다 큽니다.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, model_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        # TODO 15-9: Linear -> GELU -> Linear -> Dropout을 구성하세요.
        self.net = ...

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        # TODO: 두 LayerNorm, attention, FFN, Dropout을 정의하세요.
        raise NotImplementedError("TODO 15-9")

    def forward(self, x: Tensor, allowed: Tensor) -> tuple[Tensor, Tensor]:
        # TODO: pre-norm residual 두 단계를 구현하세요.
        raise NotImplementedError("TODO 15-9")

In [ ]:
block = DecoderBlock(model_dim, num_heads, ff_dim=96, dropout=0.0).to(device)
block_output, block_weights = block(hidden, allowed)
probe_loss = block_output.square().mean()
probe_loss.backward()
gradient_norm = block.attention.qkv.weight.grad.norm().item()
assert block_output.shape == hidden.shape
assert gradient_norm > 0
print(f"block={tuple(block_output.shape)}, qkv grad norm={gradient_norm:.6f}")

## 10) Decoder-only causal language model 조립

여러 decoder block 뒤에 final LayerNorm과 vocabulary projection(`lm_head`)을 붙입니다.
언어 모델의 logit shape은 `[B,T,V]`입니다. 위치 `t`의 logit이 다음 token `t+1`을 맞히도록
target을 한 칸 shift하며, padding target은 `ignore_index`로 loss에서 제외합니다.

In [ ]:
@dataclass(frozen=True)
class ArchitectureConfig:
    vocab_size: int
    pad_id: int
    max_length: int = 16
    model_dim: int = 32
    num_heads: int = 4
    num_layers: int = 2
    ff_dim: int = 96
    dropout: float = 0.0


class TinyCausalLM(nn.Module):
    def __init__(self, config: ArchitectureConfig):
        super().__init__()
        # TODO 15-10: embedding, block 목록, final norm, lm_head를 정의하세요.
        self.config = config
        raise NotImplementedError("TODO 15-10")

    def forward(
        self, ids: Tensor, labels: Tensor | None = None
    ) -> tuple[Tensor, Tensor | None, list[Tensor]]:
        # TODO: mask 생성 -> blocks -> logits -> shifted CE loss
        raise NotImplementedError("TODO 15-10")

In [ ]:
architecture_config = ArchitectureConfig(vocab_size=len(stoi), pad_id=pad_id)
tiny_lm = TinyCausalLM(architecture_config).to(device)
logits, lm_loss, layer_weights = tiny_lm(input_ids, labels=input_ids)
assert logits.shape == (2, 6, len(stoi))
assert lm_loss is not None and torch.isfinite(lm_loss)
assert len(layer_weights) == architecture_config.num_layers
print(f"logits={tuple(logits.shape)}, shifted loss={lm_loss.item():.4f}")

## 11) 누설(leakage) 테스트 — mask가 정말 작동하는가?

미래 위치의 token 하나만 바꿨을 때 그보다 앞선 위치의 logit은 같아야 합니다. 이 테스트는
mask 방향을 뒤집거나 softmax 뒤에 mask하는 흔한 버그를 잡습니다. Dropout은 비교를 흔들 수
있으므로 `eval()`에서 검사합니다.

In [ ]:
tiny_lm.eval()
changed = input_ids.clone()
changed[0, 3] = stoi["연결한다"]
with torch.inference_mode():
    original_logits, _, _ = tiny_lm(input_ids)
    changed_logits, _, _ = tiny_lm(changed)

# 위치 1은 미래 위치 3을 볼 수 없어야 합니다.
assert torch.allclose(original_logits[0, 1], changed_logits[0, 1], atol=1e-6)
# 바뀐 위치 이후에는 차이가 생기는 것이 정상입니다.
difference_after_change = (original_logits[0, 3] - changed_logits[0, 3]).abs().max()
print(
    "causal leakage test passed; later max difference:", difference_after_change.item()
)

## 마무리 체크

이제 다음 질문에 코드로 답할 수 있어야 합니다.

- 왜 score를 `sqrt(head_dim)`으로 나누는가?
- causal mask와 padding mask가 각각 어느 축을 막는가?
- head를 분리/병합할 때 `transpose`와 `contiguous`가 왜 필요한가?
- Pre-norm residual의 두 경로는 무엇인가?
- 언어 모델 loss에서 logit과 target을 어떻게 한 칸 이동하는가?

다음 랩에서는 이 구조와 프로젝트의 `TinyCausalTransformer`를 이용해 optimizer, AMP,
gradient accumulation, checkpoint, generation까지 하나의 학습 파이프라인으로 완성합니다.